# Build a Neo4j knowledge graph

This notebook creates a collection-scoped Neo4j knowledge graph from extracted Docling documents. It uses MaaS for embeddings and entity/relation extraction. The resulting collection is queried by the Neo4j graph-inference notebook.

## Prerequisites

Set `MAAS_API_KEY`, `MAAS_BASE_URL`, `NEO4J_URI`, and `NEO4J_PASSWORD`. Optionally set `NEO4J_USERNAME` and `NEO4J_DATABASE`. The source documents must already be available as Docling JSON files in `extracted_text_dir`.

In [ ]:
%pip install 'ai4rag[text-extraction]~={AI4RAG_VERSION}' | tail -n 1

## Configure MaaS and Neo4j

In [ ]:
import getpass
import os

from ai4rag.utils.clients.maas_client import create_maas_client

maas_api_key = os.getenv("MAAS_API_KEY") or getpass.getpass("MAAS_API_KEY: ")
maas_base_url = os.getenv("MAAS_BASE_URL") or getpass.getpass("MAAS_BASE_URL: ")
client = create_maas_client(base_url=maas_base_url, api_key=maas_api_key)

required_neo4j_vars = ("NEO4J_URI", "NEO4J_PASSWORD")
missing_neo4j_vars = [name for name in required_neo4j_vars if not os.getenv(name)]
if missing_neo4j_vars:
    raise ValueError(f"Missing Neo4j variables: {{missing_neo4j_vars}}")

## Initialize the graph store

Use the same collection name in the inference notebook. Neo4j graph retrieval is collection-isolated and supports `search_mode="graph"` only.

In [ ]:
from ai4rag.rag.embedding.openai_model import OpenAIEmbeddingModel, OpenAIEmbeddingParams
from ai4rag.rag.foundation_models.base_model import Language
from ai4rag.rag.foundation_models.openai_model import OpenAIFoundationModel
from ai4rag.rag.vector_store.config import Neo4jConfig
from ai4rag.rag.vector_store.neo4j import Neo4jGraphStore

embedding_model = OpenAIEmbeddingModel(
    client=client,
    model_id="{EMBEDDING_MODEL_ID}",
    params=OpenAIEmbeddingParams(**{EMBEDDING_PARAMS}),
)
foundation_model = OpenAIFoundationModel(
    client=client,
    model_id="{FM_MODEL_ID}",
    system_message_text="""{SYSTEM_MESSAGE}""",
    user_message_text="""{USER_MESSAGE}""",
    context_template_text="""{CONTEXT_TEXT}""",
    language=Language(**{LANGUAGE}),
)
collection_name = "{COLLECTION_NAME}"
vector_store = Neo4jGraphStore(
    embedding_model=embedding_model,
    config=Neo4jConfig.from_env(),
    collection_name=collection_name,
)

## Load documents and create the knowledge graph

Set `extracted_text_dir` to the directory containing Docling JSON files. `chunk_size` and `chunk_overlap` are passed to the KG pipeline's fixed-size text splitter.

In [ ]:
from pathlib import Path

from ai4rag.utils.docling_io import load_docling_documents

extracted_text_dir = Path("./step_outputs/extracted_text")
documents = load_docling_documents(extracted_text_dir)
if not documents:
    raise ValueError(f"No Docling documents found in {{extracted_text_dir}}")

vector_store.build_knowledge_graph_from_documents(
    documents=documents,
    model=foundation_model,
    chunk_size={CHUNK_SIZE},
    chunk_overlap={CHUNK_OVERLAP},
)
print(f"Knowledge graph created in collection: {{collection_name}}")

## Close the connection

Run this cell when indexing is complete. The graph and its collection-specific vector index remain in Neo4j.

In [ ]:
vector_store.close()